# Messy-text test — extract Ada slots from real prose (Qwen3)

Real Wikipedia prose -> Ada's REAL enricher gives gold slots -> compare
**zero-shot Qwen3** vs **in-loop fine-tuned Qwen3** (slot-F1 vs gold).

**Runtime -> GPU.** Needs two Colab secrets: `ANTHROPIC_API_KEY` (gold) and optional `HF_TOKEN`. Self-contained: clones only the PUBLIC adaL.

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes anthropic
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Runtime->GPU')


In [ ]:
%%writefile fetch.py
"""
fetch.py — acquire REAL messy prose for the messy-text test.

Pulls live Wikipedia summaries (real, paraphrased, implicit prose) for a list of
titles -> messy.jsonl {title, text}. This is the "messy" the clean rendered facts
never had. No API key (Wikipedia REST is public).

Run:  python fetch.py --titles titles.txt --n 500 --out messy.jsonl
      python fetch.py --n 50            # uses a built-in sample if no titles file
"""

import json, time, argparse, urllib.request, urllib.parse

SAMPLE = ["Caffeine", "Aspirin", "Sodium chloride", "Glucose", "Penicillin",
          "Ethanol", "Benzene", "Insulin", "Chlorophyll", "Ozone", "Graphite",
          "Ammonia", "Acetic acid", "Dopamine", "Serotonin", "Methane",
          "Sulfuric acid", "Cholesterol", "Adrenaline", "Citric acid"]


def fetch_summary(title):
    t = urllib.parse.quote(title.replace(" ", "_"))
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{t}"
    req = urllib.request.Request(url, headers={"User-Agent": "ada-messy-test/0.1"})
    d = json.load(urllib.request.urlopen(req, timeout=20))
    if d.get("type", "").endswith("disambiguation"):
        return None
    return d.get("extract") or None


def main(titles_path, n, out):
    if titles_path:
        titles = [l.strip() for l in open(titles_path) if l.strip()][:n]
    else:
        titles = SAMPLE[:n]
    ok = 0
    with open(out, "w") as fo:
        for t in titles:
            try:
                text = fetch_summary(t)
            except Exception as e:
                text = None
            if text and len(text) > 80:
                fo.write(json.dumps({"title": t, "text": text}) + "\n"); ok += 1
            time.sleep(0.1)  # be polite
    print(f"wrote {ok}/{len(titles)} messy summaries -> {out}")
    if ok:
        ex = json.loads(open(out).readline())
        print("sample:", ex["title"], "->", ex["text"][:200])


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--titles", default=None)
    ap.add_argument("--n", type=int, default=50)
    ap.add_argument("--out", default="messy.jsonl")
    a = ap.parse_args()
    main(a.titles, a.n, a.out)


In [ ]:
%%writefile enrich.py
"""
enrich.py — gold labels for the messy test, via Ada's REAL enricher.

For each messy summary, call adaL's UniversalEnricher (LLM-at-write-time) to map
the prose -> Ada's 7-layer/33-role schema. That enricher output IS the gold (it
is, by definition, what "correct extraction" means for Ada). -> gold.jsonl
{title, text, slots}.

REQUIRES: ANTHROPIC_API_KEY (the enricher calls the API), and the adaL repo
(auto-cloned). This is the messy task's labeling step — can't run without a key.

Run:  ANTHROPIC_API_KEY=... python enrich.py --in messy.jsonl --out gold.jsonl
"""

import os, sys, json, argparse, subprocess

ADAL = "/tmp/adaL"


def ensure_adal():
    if not os.path.exists(ADAL):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/glyphh-ai/adaL", ADAL],
                       check=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    sys.path.insert(0, ADAL)


def main(inp, out):
    if not os.environ.get("ANTHROPIC_API_KEY"):
        sys.exit("ANTHROPIC_API_KEY not set — the enricher needs it to label gold.")
    ensure_adal()
    from ada.cognitive.universal import UniversalEnricher   # Ada's REAL enricher
    enr = UniversalEnricher()                                # model from ADA_MODEL/default
    n = 0
    with open(inp) as fi, open(out, "w") as fo:
        for line in fi:
            rec = json.loads(line)
            try:
                slots = enr.enrich(rec["text"])              # prose -> {layer:{role:value}}
            except Exception as e:
                print("skip", rec["title"], e); continue
            fo.write(json.dumps({"title": rec["title"], "text": rec["text"],
                                 "slots": slots}) + "\n")
            n += 1
            if n % 50 == 0:
                print(f"enriched {n}...")
    print(f"wrote {n} gold (text, slots) pairs -> {out}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--in", dest="inp", default="messy.jsonl")
    ap.add_argument("--out", default="gold.jsonl")
    a = ap.parse_args()
    main(a.inp, a.out)


In [ ]:
%%writefile eval_messy.py
"""
eval_messy.py — the real-world messy test: extract Ada slots from MESSY prose.

Unlike the clean task (token-tag a value that appears verbatim), here the slot
value is often paraphrased/implicit in the prose -> the model must GENERATE the
structured slots, i.e. be Ada's enricher. We score generated slots vs gold
(Ada's enricher output) by slot-level F1.

Systems compared:
  - zero-shot Qwen3   : prompt the base model to emit the schema (no fine-tune)
  - in-loop Qwen3     : the model fine-tuned to emit Ada slots (frontier/finetune_qwen)
The question: does in-loop training make the model a BETTER enricher on messy
text than prompting alone — and how close to gold?

The slot-F1 metric is validated on CPU here (self-test). The Qwen3 generations
need a GPU (Colab).

Run (CPU self-test of the metric):   python eval_messy.py --selftest
Run (GPU eval, after gold.jsonl):    python eval_messy.py --gold gold.jsonl
"""

import json, argparse


def flat(slots):
    """{layer:{role:value}} -> set of 'layer.role=value' (lowercased)."""
    out = set()
    for layer, roles in (slots or {}).items():
        for role, val in (roles or {}).items():
            if val:
                out.add(f"{layer}.{role}={str(val).strip().lower()}")
    return out


def slot_f1(pred, gold):
    P, G = flat(pred), flat(gold)
    if not P and not G:
        return 1.0, 1.0, 1.0
    tp = len(P & G)
    prec = tp / len(P) if P else 0.0
    rec = tp / len(G) if G else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return prec, rec, f1


def selftest():
    gold = {"entity": {"name": "Caffeine", "kind": "stimulant"},
            "relational": {"predicate": "blocks", "object": "adenosine"}}
    perfect = gold
    partial = {"entity": {"name": "caffeine", "kind": "drug"},          # kind wrong
               "relational": {"predicate": "blocks", "object": "adenosine"}}
    empty = {}
    for name, pred in (("perfect", perfect), ("partial", partial), ("empty", empty)):
        print(f"  {name:8} P/R/F1 = %.2f %.2f %.2f" % slot_f1(pred, gold))
    assert slot_f1(perfect, gold)[2] == 1.0
    assert 0 < slot_f1(partial, gold)[2] < 1.0
    assert slot_f1(empty, gold)[2] == 0.0
    print("slot-F1 metric self-test PASSED.")


# ---- GPU eval (Colab): prompt/generate Ada slots from messy text ----
PROMPT = (
    "Extract a fixed-schema record from the text. Output JSON with only these "
    "layers/roles when present: entity{name,kind,subkind}, perceptual{color,"
    "size,shape,...}, spatial{location,origin}, temporal{time,age,era}, "
    "relational{predicate,object}, quantitative{count,magnitude,unit}. "
    "Text:\n{t}\nJSON:"
)

def gpu_eval(gold_path, model_name):
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(model_name)
    m = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto",
                                             torch_dtype=torch.bfloat16)
    rows = [json.loads(l) for l in open(gold_path)]
    f1s = []
    for r in rows:
        msg = PROMPT.replace("{t}", r["text"][:1200])
        ids = tok(msg, return_tensors="pt").to(m.device)
        out = m.generate(**ids, max_new_tokens=200, do_sample=False)
        gen = tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
        try:
            pred = json.loads(gen[gen.find("{"):gen.rfind("}") + 1])
        except Exception:
            pred = {}
        f1s.append(slot_f1(pred, r["slots"])[2])
    print(f"zero-shot {model_name}: mean slot-F1 vs Ada gold = {sum(f1s)/len(f1s):.3f}  (n={len(f1s)})")
    print("(compare to an in-loop fine-tuned model trained to emit the schema)")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--selftest", action="store_true")
    ap.add_argument("--gold", default="gold.jsonl")
    ap.add_argument("--model", default="Qwen/Qwen3-8B")
    a = ap.parse_args()
    if a.selftest:
        selftest()
    else:
        gpu_eval(a.gold, a.model)


In [ ]:
%%writefile finetune_messy.py
"""
finetune_messy.py — the in-loop arm of the messy test: distill Ada's enricher.

The messy task is GENERATION (prose -> Ada-slot JSON), not tagging. So "in-loop"
here = LoRA fine-tune Qwen3 to emit the schema given messy text, trained on
(gold.text -> gold.slots). Then compare, on a held-out split:

  - zero-shot  : base Qwen3, prompted (no fine-tune)
  - in-loop    : the fine-tuned model
  - ceiling    : Ada's own enricher (gold = 1.0 by definition)

scored by slot-F1 vs gold.

PRE-REGISTERED BAR (lock before running):
  in-loop slot-F1 >= zero-shot + 0.10  AND  in-loop >= 0.60

GPU only (Colab). Not run in the CPU sandbox.

Run:  python finetune_messy.py --gold gold.jsonl --model Qwen/Qwen3-8B --steps 600
"""

import json, argparse, random
from eval_messy import slot_f1, PROMPT      # reuse the validated metric + prompt

BAR_DELTA, BAR_ABS = 0.10, 0.60


def target_json(slots):
    return json.dumps(slots, separators=(",", ":"))


def make_examples(tok, rows, max_in=900, max_tgt=200):
    exs = []
    for r in rows:
        p = tok(PROMPT.replace("{t}", r["text"][:1200]), truncation=True, max_length=max_in)["input_ids"]
        t = tok(target_json(r["slots"]), truncation=True, max_length=max_tgt)["input_ids"] + [tok.eos_token_id]
        ids = p + t
        labels = [-100] * len(p) + t            # train only on the JSON target
        exs.append({"ids": ids, "labels": labels})
    return exs


def collate(batch, pad):
    import torch
    T = max(len(b["ids"]) for b in batch)
    ids = torch.tensor([b["ids"] + [pad] * (T - len(b["ids"])) for b in batch])
    lab = torch.tensor([b["labels"] + [-100] * (T - len(b["labels"])) for b in batch])
    am = (ids != pad).long()
    return ids, am, lab


def generate_slots(model, tok, text, dev):
    import torch
    msg = PROMPT.replace("{t}", text[:1200])
    enc = tok(msg, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=200, do_sample=False,
                             pad_token_id=tok.pad_token_id)
    gen = tok.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True)
    try:
        return json.loads(gen[gen.find("{"):gen.rfind("}") + 1])
    except Exception:
        return {}


def eval_split(model, tok, rows, dev):
    return sum(slot_f1(generate_slots(model, tok, r["text"], dev), r["slots"])[2]
               for r in rows) / max(1, len(rows))


def main(gold, model_name, steps, lr):
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    rows = [json.loads(l) for l in open(gold)]
    random.Random(0).shuffle(rows)
    k = int(0.85 * len(rows)); tr, va = rows[:k], rows[k:]
    print(f"{len(rows)} gold pairs -> train {len(tr)} / val {len(va)}  | {model_name}")

    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=torch.bfloat16) if dev == "cuda" else None
    model = AutoModelForCausalLM.from_pretrained(
        model_name, quantization_config=qcfg, device_map="auto" if dev == "cuda" else None,
        torch_dtype=torch.bfloat16 if dev == "cuda" else torch.float32)

    # 1) zero-shot baseline (before any fine-tuning)
    zs = eval_split(model, tok, va, dev)
    print(f"[zero-shot] slot-F1 vs Ada gold = {zs:.3f}")

    # 2) in-loop: LoRA fine-tune to emit the schema
    if dev == "cuda": model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
    exs = make_examples(tok, tr)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    model.train()
    for s in range(1, steps + 1):
        b = [exs[random.randrange(len(exs))] for _ in range(4)]
        ids, am, lab = (t.to(dev) for t in collate(b, tok.pad_token_id))
        out = model(input_ids=ids, attention_mask=am, labels=lab)
        out.loss.backward(); opt.step(); opt.zero_grad()
        if s % 100 == 0 or s == 1:
            print(f"step {s:4d}  loss {out.loss.item():.3f}")

    model.eval()
    il = eval_split(model, tok, va, dev)

    print(f"\n=== MESSY TEST (slot-F1 vs Ada enricher gold) ===")
    print(f"  zero-shot Qwen3 : {zs:.3f}")
    print(f"  in-loop  Qwen3  : {il:.3f}")
    print(f"  Ada enricher    : 1.000  (ceiling/gold)")
    ok = (il >= zs + BAR_DELTA) and (il >= BAR_ABS)
    print(f"\nPRE-REGISTERED BAR (in-loop >= zero-shot+{BAR_DELTA} and >= {BAR_ABS}): "
          f"{'PASS' if ok else 'FAIL'}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--gold", default="gold.jsonl")
    ap.add_argument("--model", default="Qwen/Qwen3-8B")
    ap.add_argument("--steps", type=int, default=600)
    ap.add_argument("--lr", type=float, default=2e-4)
    a = ap.parse_args()
    main(a.gold, a.model, a.steps, a.lr)


In [ ]:
# real messy prose from adaL's crawled chemistry titles (public clone)
!git clone --depth 1 https://github.com/glyphh-ai/adaL /tmp/adaL
!python fetch.py --titles /tmp/adaL/benchmark/curriculum/agi/chemistry/phase1_output/corpus_titles.txt --n 1500


In [ ]:
# gold via Ada's REAL enricher  (~$0.0011/text -> ~$1.6 for 1500)
import os
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
!python eval_messy.py --selftest        # metric sanity (CPU)
!python enrich.py --in messy.jsonl --out gold.jsonl


In [ ]:
# the verdict: zero-shot vs in-loop fine-tuned, slot-F1 vs Ada gold
!python finetune_messy.py --gold gold.jsonl --model Qwen/Qwen3-8B --steps 600


## Read it
Pre-registered bar: **in-loop slot-F1 >= zero-shot +0.10 AND >= 0.60**.
If PASS, in-loop fine-tuning makes Qwen3 a meaningfully better enricher on
messy prose than prompting alone — the messy-text claim holds. If FAIL, it
doesn't, and we say so.